We forked xbatcher to let us filter patches. So you have to pip install from git.

In [ ]:
!pip install -q git+https://github.com/s-kganz/xbatcher.git@patch_filter_resample

In [ ]:
import torch
import xarray as xr
import xbatcher
import numpy as np
import matplotlib.pyplot as plt

# If we have gpu available, use it
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()
torch.set_default_device(device)

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Prepare data

In [ ]:
ds = xr.open_zarr("../data_working/westmort.zarr/").compute()

# Identify all damage variables and collapse them together
target_vars = list(filter(lambda x: x.endswith("_target"), ds.variables.keys()))
print(target_vars)
damage = ds[target_vars].to_dataarray(dim="variable").mean(dim="variable")

# Create a mask from basal area
ba_vars = list(filter(lambda x: x.endswith("_ba"), ds.variables.keys()))
ba_mask = ds[ba_vars].to_dataarray(dim="variable").sum(dim="variable") > 0

# Mask damage pixels with no hosts
damage = damage.where(ba_mask).transpose("time", "y", "x")

In [ ]:
del ds

Split data into train/valid/test segments.

In [ ]:
years = damage.time

train_years = years[16:]
valid_years = years[:8]
test_years = years[8:16]

print("Training years:", train_years.dt.year.data)
print("Validation years:", valid_years.dt.year.data)
print("Testing years:", test_years.dt.year.data)

ds_train = damage.sel(time=train_years)
ds_valid = damage.sel(time=valid_years)
ds_test  = damage.sel(time=test_years)

Set up windowing parameters.

In [ ]:
# This is the full slice for each training instance.
# But the prediction label is the central 8x8 window
# in the last time step. This section must be fully
# Non-NA to be accepted.
WINDOW = dict(x=16, y=16, time=5)
OVERLAP = dict(x=8, y=8, time=4)

def patch_filter(ds: xr.DataArray, selector: dict) -> bool:
    patch = ds.isel(**selector)
    last_step = patch.isel(time=-1)
    # Get the middle
    x_size = WINDOW["x"] // 4
    y_size = WINDOW["y"] // 4
    last_step_middle = last_step.isel(x=slice(x_size, -x_size), y=slice(y_size, -y_size))
    no_nulls = last_step_middle.isnull().astype(np.float32).sum() == 0
    has_damage = last_step_middle.mean() > 0
    return no_nulls and has_damage

In [ ]:
bgen = xbatcher.BatchGenerator(
    ds_train,
    input_dims=WINDOW,
    input_overlap=OVERLAP,
    filter_fn=patch_filter
)
print(len(bgen))

Set up data pipeline to go from batch gen -> DataLoader.

In [ ]:
from torch.utils.data import DataLoader, Dataset

class BatchGenDataset(Dataset):
    def __init__(self, bgen):
        self.bgen = bgen

    def __len__(self):
        return len(self.bgen)

    def __getitem__(self, idx):
        return self.bgen[idx]

def collator(patches, dtype=torch.float32):
    # Patch dimensions are (time, y, x)
    X = torch.stack([
        # Exclude last time step
        torch.tensor(
            patch.values[:-1, np.newaxis, ...],
            dtype=dtype
        )
        for patch in patches
    ])

    # Replace nan cells with zeros, scale from 0-1
    X = torch.nan_to_num(X)

    # Snip out the middle of the last time step, and scale from 0-1
    y = torch.stack([
        torch.tensor(patch.values[-1, WINDOW["y"]//4:-WINDOW["y"]//4, WINDOW["x"]//4:-WINDOW["x"]//4], dtype=dtype)
        for patch in patches
    ])/100

    return X, y


my_dataloader = DataLoader(
    BatchGenDataset(bgen),
    batch_size=16,
    collate_fn=collator,
    generator=torch.Generator(device=device),
    shuffle=True,
    drop_last=True
)

In [ ]:
X, y = next(iter(my_dataloader))
print(X.shape, y.shape)

Define model structure.

In [ ]:
from convlstm import DamageConvLSTM
import pytorch_lightning as pl
import torch.nn.functional as F
import torchmetrics

class Model(pl.LightningModule):
    def __init__(self, input_shape: int, input_dim=1, hidden_dim=16, batch_first=True, **kwargs):
        super(Model, self).__init__()
        self.dclstm = DamageConvLSTM(input_dim=input_dim, hidden_dim=hidden_dim, batch_first=batch_first, **kwargs)
        self.conv = torch.nn.Conv2d(1, 1, kernel_size=input_shape//2+1)

        self.train_nrmse = torchmetrics.regression.NormalizedRootMeanSquaredError()
        self.train_r    = torchmetrics.regression.PearsonCorrCoef()
        self.train_spear = torchmetrics.regression.SpearmanCorrCoef()

        self.valid_nrmse = torchmetrics.regression.NormalizedRootMeanSquaredError()
        self.valid_r    = torchmetrics.regression.PearsonCorrCoef()
        self.valid_spear = torchmetrics.regression.SpearmanCorrCoef()

    def forward(self, X):
        X = self.dclstm(X)
        X = self.conv(X)
        X = X.squeeze()
        return X

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.005)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.2, patience=3, min_lr=5e-5)
        return {"optimizer": optimizer, "lr_scheduler": scheduler, "monitor": "valid_loss"}

    def _get_loss(self, y, y_hat):
        loss = F.mse_loss(y*100, y_hat*100)
        return loss

    def training_step(self, batch, batch_idx):
        X, y = batch
        y_hat = self.forward(X)

        loss = self._get_loss(y, y_hat)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)

        self.train_nrmse(y, y_hat)
        self.log("train_nrmse", self.train_nrmse, on_epoch=True, on_step=False)

        self.train_r(y.view(-1), y_hat.view(-1))
        self.log("train_r", self.train_r, on_epoch=True, on_step=False)

        self.train_spear(y.view(-1), y_hat.view(-1))
        self.log("train_spear", self.train_spear, on_epoch=True, on_step=False)
            
        return loss

    def validation_step(self, batch, batch_idx):
        X, y = batch
        y_hat = self.forward(X)

        loss = self._get_loss(y, y_hat)
        self.log("valid_loss", loss, prog_bar=True, on_epoch=True, on_step=False)

        self.valid_nrmse(y, y_hat)
        self.log("valid_nrmse", self.valid_nrmse, on_epoch=True, on_step=False)

        self.valid_r(y.view(-1), y_hat.view(-1))
        self.log("valid_r", self.valid_r, on_epoch=True, on_step=False)

        self.valid_spear(y.view(-1), y_hat.view(-1))
        self.log("valid_spear", self.valid_spear, on_epoch=True, on_step=False)

    def test_step(self, batch, batch_idx):
        X, y = batch
        y_hat = self.forward(X)

        loss = self._get_loss(y, y_hat)
        self.log("test_loss", loss, prog_bar=True, on_epoch=True, on_step=False)

Run an experiment!

In [ ]:
train_bgen = xbatcher.BatchGenerator(
    ds_train,
    input_dims=WINDOW,
    input_overlap=OVERLAP,
    filter_fn=patch_filter
)

valid_bgen = xbatcher.BatchGenerator(
    ds_valid,
    input_dims=WINDOW,
    input_overlap=OVERLAP,
    filter_fn=patch_filter
)

print("N training:", len(train_bgen))
print("N valid:", len(valid_bgen))

train_dl = DataLoader(
    BatchGenDataset(train_bgen),
    batch_size=16,
    collate_fn=collator,
    generator=torch.Generator(device=device),
    shuffle=True,
    drop_last=True
)

valid_dl = DataLoader(
    BatchGenDataset(valid_bgen),
    batch_size=16,
    collate_fn=collator,
    generator=torch.Generator(device=device),
    shuffle=False,
    drop_last=True
)

In [ ]:
m = Model(
    input_shape=16,
    input_dim=1,
    hidden_dim=8,
    batch_first=True,
    dropout=0.3,
    kernel_size=(3, 3),
    num_layers=2
)

trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    max_epochs=30,
    callbacks=[
        pl.callbacks.EarlyStopping("valid_loss", min_delta=1e-4, patience=10),
        pl.callbacks.ModelCheckpoint(),
        pl.callbacks.LearningRateMonitor(logging_interval="epoch")
    ]
)

trainer.fit(m, train_dl, valid_dl)

In [ ]:
# Make a random sample of 100 background instances. This is generally
# sufficient to get a good SHAP estimate.
from shap import DeepExplainer

shap_samples = np.random.randint(low=0, high=len(train_bgen), size=200)
shap_X, _ = collator([train_bgen[int(i)] for i in shap_samples])
shap_X = shap_X.to(device)

In [ ]:
# SHAP is only possible on a scalar output, so we have to collapse the 8x8
# CNN output to a scalar.
class ShapAdapter(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module):
        super(ShapAdapter, self).__init__()
        self.base_model = base_model

    def forward(self, X):
        X = self.base_model(X)
        # SHAP does not support logit space :(
        #X = torch.special.logit(X, eps=1e-5)
        X = torch.mean(X, dim=[1, 2]).unsqueeze(-1)
        return X

m_adapt = ShapAdapter(m).to(device)
out = m_adapt(shap_X)

In [ ]:
de = DeepExplainer(m_adapt, shap_X)

In [ ]:
shap_values = de.shap_values(shap_X, check_additivity=False)

In [ ]:
print(shap_values.shape)

In [ ]:
mean_over_space = np.mean(np.abs(shap_values), axis=(0, 1, 2, -1))

In [ ]:
import matplotlib
from matplotlib import pyplot as plt

fig, ax = plt.subplots()

window_y = shap_values.shape[-3]
window_x = shap_values.shape[-2]
cell_size_km = 3

x_ticks = np.arange(0, window_x, 2)
y_ticks = np.arange(0, window_y, 2)
x_offset = (x_ticks - (window_x//2)) * cell_size_km
y_offset = (y_ticks - (window_y//2)) * cell_size_km * -1

patch = matplotlib.patches.Rectangle((3.5, 3.5), 8, 8, edgecolor="red", facecolor="none", linewidth=2, ls="--")
im = ax.imshow(mean_over_space)
ax.add_patch(patch)

ax.set_xticks(x_ticks-0.5, labels=x_offset)
ax.set_yticks(y_ticks-0.5, labels=y_offset)

fig.colorbar(im, label="Absolute value of SHAP")
plt.ylabel("Kilometers north of window center")
plt.xlabel("Kilometers east of window center")
plt.show()

In [ ]:
shap_values_by_time = np.mean(np.abs(shap_values), axis=0).reshape(4, -1)
print(shap_values_by_time.shape)

In [ ]:
plt.boxplot(np.abs(shap_values_by_time).T)
plt.xticks(ticks=[1, 2, 3, 4], labels=[4, 3, 2, 1])
plt.xlabel("Years prior to prediction")
plt.ylabel("Absolute value of SHAP")
plt.show()